# Working with Reddit

## Lecture objectives
1. Demonstrate how to scrape Reddit data using their API

We reviewed topic modeling in the previous lecture. Here and in the next lecture, we'll focus on another common Natural Language Processing tool: sentiment analysis. In short, sentiment analysis tries to understand whether a snippet of text (e.g. a tweet, a review, or a sentence from an article) is positive, negative, or neutral.

We'll apply sentiment analysis to some Reddit data on public transportation, using the [PRAW](https://praw.readthedocs.io/en/stable/) library.

If you want to access the Reddit data yourself, you'll need to:

(1) sign up for a Reddit account (free)

(2) create a client id and client secret. It's also free, and takes about 5 minutes. [Follow the second part of these instructions.](https://cs205uiuc.github.io/guidebook/python/reddit-api.html) You can get to the apps tab here: https://www.reddit.com/prefs/apps.

In earlier versions of this course, I used Twitter. However, academic access to the Twitter API is no longer free. For a thorough treatment of obtaining, analyzing, and interpreting Twitter data, check out [*Twitter as Data*](https://www.cambridge.org/core/elements/twitter-as-data/27B3DE20C22E12E162BFB173C5EB2592) by Prof. Zachary Steinert-Threlkeld here in the Luskin School of Public Affairs.

## Using the Reddit API
The `praw` library provides easy access to Reddit. You can enter your credentials here, or just follow along for the time being.

In [1]:
import praw

# enter your own client_id and client_secret
client_id = 'wOXSJyd2tAX035dsqu01VQ'
client_secret = 'S3JXQi71AUATOMrrYH9MGac0ZPa1Lw'
# this identifies you, but can be any string
user_agent = 'scraper by u/adammb_ucla'

reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent,
)

Version 7.7.1 of praw is outdated. Version 7.8.1 was released Friday October 25, 2024.


Now we have our `reddit` object that has several methods.

For example, we can get new posts, and loop over them. Here's we'll get the latest 10 from the Urban Planning subreddit.

In [2]:
# subredditとは、掲示板のこと
# urbanplanningの掲示板にされている投稿を探す
for submission in reddit.subreddit('urbanplanning').new(limit=10):
    print(submission.title)

Planners, what are some small things that can be done to help our communities?
If (primarily) American Urbanists are pushing Japan-style zoning to end issues like the loneliness epidemic in the states, then what is the Urbanist diagnosis/solution for the Japanese loneliness epidemic?
If you could redo New York from scratch, what would you change?
Colorado officials ​p​lan Denver-Fort Collins rail service ​by 2029
Tailgates at City Hall? Rethinking How We Engage with Local Urban Planning
Shops make a city great
The Real Reason You're Sitting in Traffic | Streetcraft
The 2 Car Garage—Why it Messes Up Houses Today
Intent of the Code
Resources or experience with good form based codes?


The `submission` object provides access to the more detailed post information.

In [3]:
# submissionには色々なargumentが入っている
submission?

Type:           Submission
String form:    1kvdr7l
File:           /opt/anaconda3/envs/uds/lib/python3.12/site-packages/praw/models/reddit/submission.py
Docstring:     
A class for submissions to Reddit.

.. include:: ../../typical_attributes.rst

========================== =========================================================
Attribute                  Description
========================== =========================================================
``author``                 Provides an instance of :class:`.Redditor`.
``author_flair_text``      The text content of the author's flair, or ``None`` if
                           not flaired.
``clicked``                Whether or not the submission has been clicked by the
                           client.
``comments``               Provides an instance of :class:`.CommentForest`.
``created_utc``            Time the submission was created, represented in `Unix
                           Time`_.
``distinguished``          Whether or not 

Let's look at the comments within the last submission.

In [4]:
# submission（投稿）に対する返信コメントの数も出せる
submission.num_comments

4

In [5]:
# コメントの中身も見れる
for c in submission.comments:
    print(c.body)

https://formbasedcodes.org/
To be honest with you it doesn't sound like they are setting you up for any kind of success. Form based codes have been an idea for a while, and I don't think any place has really done them successfully--at least in the way people envision the way they are supposed to function.

The pre-made towns like sunrise or discovery (wherever they filmed the Truman Show) worked because it was a single developer doing everything from the beginning. Other cities that have tried form based (LA, Denver, and recently i saw Cleveland was trying) always seem to start out with big visions but then get bogged down by big complexity.

In my opinion, form-based zones are always going to fail because most cities are not actually using zoning to regulate how things look as much as who is allowed to do what, and "pure" form-based codes largely give that power up. To date I've never seen a form-based code that ultimately didn't just become the design standards chapter of an already 

Each comment `c` also has further attributes. We used `body` to get the text of the comment, but there are also timestamps, author details, and so on.

In [6]:
c?

Type:           Comment
String form:    mufb0ku
File:           /opt/anaconda3/envs/uds/lib/python3.12/site-packages/praw/models/reddit/comment.py
Docstring:     
A class that represents a Reddit comment.

.. include:: ../../typical_attributes.rst

================= =================================================================
Attribute         Description
================= =================================================================
``author``        Provides an instance of :class:`.Redditor`.
``body``          The body of the comment, as Markdown.
``body_html``     The body of the comment, as HTML.
``created_utc``   Time the comment was created, represented in `Unix Time`_.
``distinguished`` Whether or not the comment is distinguished.
``edited``        Whether or not the comment has been edited.
``id``            The ID of the comment.
``is_submitter``  Whether or not the comment author is also the author of the
                  submission.
``link_id``       The submission

Let's get the submission and all the comments from the latest 100 posts from three transit subreddits: LA Metro, BART, and NYC rail.

We'll define a function that takes the subreddit name, and returns all of these comments in a single list.

In [7]:
def get_reddit(subreddit_name):
    r_list = []
    # subredditごとに内容を調べる
    for submission in reddit.subreddit(subreddit_name).new(limit=50):
        r_list.append(submission.title)
        r_list += [c.body for c in submission.comments] 

    print('Retrieved {} comments for {}'.format(len(r_list), subreddit_name))
    return r_list

la_metro = get_reddit('LAMetro')

Retrieved 417 comments for LAMetro


Let's do the same for the other two agencies.

In [8]:
bart = get_reddit('Bart')
nyc_rail = get_reddit('Nycrail')

Retrieved 576 comments for Bart
Retrieved 379 comments for Nycrail


Now let's save these comments to a file. We'll use a pickle, which as we've seen in earlier modules, can save most Python objects in their original format. (We could also have looped over the list of comments and saved them as text.)

In [9]:
import pickle
with open('../data/reddit/la_metro.pickle', 'wb') as f:
    pickle.dump(la_metro, f)
with open('../data/reddit/bart.pickle', 'wb') as f:
    pickle.dump(bart, f)
with open('../data/reddit/nyc_rail.pickle', 'wb') as f:
    pickle.dump(nyc_rail, f)

We'll pick up these data in the next lecture and see how to analyze the sentiment of the Reddit posts.

But we have only scratched the surface of the PRAW library. Explore the documentation for examples of how to filter your results (e.g. you could search for all posts within a subreddit that mention "rent" or "eviction"), access the number of upvotes, and more.

<div class="alert alert-block alert-info">
<h3>Key Takeaways</h3>
<ul>
  <li>Reddit has a powerful API that is relatively easy to use.</li>
  <li>Reddit is not representative. Whether that matters depends on your particular project and use case.</li>
</ul>
</div>